# 05 · Benchmark the PDF parsers

> **Run order.** This notebook is step 5 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.


Most parser comparisons measure characters extracted per second. That is the
wrong metric here: a parser that extracts plenty of prose but mangles the digits
inside a financial table is worse than useless.

So the headline metric is **oracle recall** — of the financial facts we already
know to be true, what fraction can be located in the text this parser produced?
It predicts how many benchmark questions notebook 06 can generate, and it costs
**zero LLM calls**.

In [ ]:
# The project is installed in editable mode by `uv sync`, so `analyst` imports
# directly. Nothing here manipulates sys.path.
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

import time
from decimal import Decimal
from pathlib import Path

from sqlalchemy import select
from analyst.db import session_scope
from analyst.models import Document, Fact
from analyst.numfmt import candidate_strings, normalise

MIN_ABS_VALUE = Decimal(10**7)

def parse_pymupdf(path, max_pages):
    import pymupdf
    t0 = time.perf_counter()
    doc = pymupdf.open(path)
    n = doc.page_count if max_pages is None else min(doc.page_count, max_pages)
    texts = [doc[i].get_text() for i in range(n)]
    doc.close()
    return texts, time.perf_counter() - t0

def parse_pdfplumber(path, max_pages):
    import pdfplumber
    t0 = time.perf_counter()
    texts = []
    with pdfplumber.open(path) as pdf:
        n = len(pdf.pages) if max_pages is None else min(len(pdf.pages), max_pages)
        for i in range(n):
            texts.append(pdf.pages[i].extract_text() or "")
    return texts, time.perf_counter() - t0

PARSERS = {"pymupdf": parse_pymupdf, "pdfplumber": parse_pdfplumber}

def oracle_recall(page_texts, facts):
    whole = normalise("\n".join(page_texts))
    found = 0
    for _concept, value in facts:
        cands = [normalise(c) for c in candidate_strings(value)]
        if any(c in whole for c in cands):
            found += 1
    return found

def load_facts(ticker):
    with session_scope() as s:
        rows = s.execute(select(Fact.concept, Fact.value).where(Fact.ticker == ticker)
                         .where(Fact.unit.in_(("INR", "USD"))).order_by(Fact.concept)).all()
    seen, out = set(), []
    for concept, value in rows:
        if concept in seen or abs(value) < MIN_ABS_VALUE:
            continue
        seen.add(concept)
        out.append((concept, value))
    return out

with session_scope() as s:
    targets = [(d.ticker, d.fiscal_year, Path(d.local_path))
               for d in s.execute(select(Document).order_by(Document.ticker)).scalars().all()]
print(f"{len(targets)} documents to benchmark")

## Head-to-head on identical 60-page subsets

Same pages, same facts, same machine. The only variable is the parser.

In [ ]:
rows = []
for ticker, _fy, path in targets:
    facts = load_facts(ticker)
    for name, fn in PARSERS.items():
        texts, seconds = fn(path, 60)
        rows.append({
            "document": path.stem, "parser": name, "pages": len(texts),
            "seconds": round(seconds, 1), "kchars": round(sum(map(len, texts)) / 1000),
            "oracle_found": oracle_recall(texts, facts), "oracle_total": len(facts),
        })
subset = pd.DataFrame(rows)
slowest = subset.groupby("document")["seconds"].transform("max")
subset["speedup_vs_slowest"] = (slowest / subset["seconds"]).round(1)
subset

In [ ]:
verdict = subset.groupby("parser").agg(
    total_seconds=("seconds", "sum"),
    total_kchars=("kchars", "sum"),
    oracle_found=("oracle_found", "sum"),
).round(1)
print(verdict.to_string())
faster = verdict.loc["pdfplumber", "total_seconds"] / verdict.loc["pymupdf", "total_seconds"]
print(f"\nPyMuPDF is {faster:.0f}x faster for essentially the same text and the same recall.")

## Full-document oracle recall

⚠️ This is a **raw** match rate and includes false positives — a 4-digit figure can collide by chance across 300 pages. Notebook 06 fixes that by requiring the number to sit beside its concept label. Treat the number below as a ceiling, not a score.

In [ ]:
rows = []
for ticker, _fy, path in targets:
    facts = load_facts(ticker)
    texts, seconds = parse_pymupdf(path, None)
    found = oracle_recall(texts, facts)
    rows.append({"document": path.stem, "pages": len(texts), "seconds": round(seconds, 1),
                 "found": found, "total": len(facts),
                 "raw_recall": round(found / len(facts), 3) if facts else None})
pd.DataFrame(rows)